# Assignment 5 — Option A: "Ask My Resume" RAG Chatbot
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Sadaf Sarbazi  
**Date:** May 2026  
**Option:** A — Resume RAG Chatbot  
**API Path:** Free (HuggingFace Inference API + sentence-transformers)  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Document Loading](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Retrieval Chain](#5-retrieval)
6. [Prompt Engineering](#6-prompting)
7. [Evaluation](#7-evaluation)

---
<a id='1-setup'></a>
## 1. Setup and Imports

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / '.env')

HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found — check your .env file'
print('HF_TOKEN loaded successfully')

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from huggingface_hub import InferenceClient

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
EMBED_MODEL = 'all-MiniLM-L6-v2'
DATA_DIR = Path('..') / 'data'

print(f'LLM: {MODEL_ID}')
print(f'Embedding model: {EMBED_MODEL}')
print(f'Data directory: {DATA_DIR.resolve()}')

---
<a id='2-loading'></a>
## 2. Document Loading

Loading 3 career documents from `data/`:
- `Resume.pdf` — professional resume
- `SOQ.pdf` — statement of qualifications / cover letter
- `github_projects.txt` — GitHub portfolio and project summaries

In [ ]:
def load_pdf(filepath):
    reader = PdfReader(filepath)
    return '\n'.join(page.extract_text() or '' for page in reader.pages)

def load_txt(filepath):
    return Path(filepath).read_text(encoding='utf-8')

def load_all_documents(data_dir):
    docs = []
    for f in sorted(Path(data_dir).iterdir()):
        if f.suffix == '.pdf':
            text = load_pdf(f)
        elif f.suffix == '.txt':
            text = load_txt(f)
        else:
            continue
        if text.strip():
            docs.append({'text': text, 'source': f.name})
    return docs

documents = load_all_documents(DATA_DIR)

print(f'Documents loaded: {len(documents)}')
for doc in documents:
    print(f"  - {doc['source']}: {len(doc['text'])} characters")
    print(f"    Preview: {doc['text'][:150].strip()}...")
    print()

---
<a id='3-chunking'></a>
## 3. Text Chunking

Comparing two chunking strategies:
- **Strategy 1 — Fixed-size:** Split by character count with overlap
- **Strategy 2 — Paragraph-aware:** Split on double newlines (natural paragraph breaks), then cap long paragraphs

In [ ]:
# ── Strategy 1: Fixed-size chunking ──

def chunk_fixed(documents, chunk_size=400, overlap=50):
    chunks = []
    for doc in documents:
        text = doc['text']
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end].strip()
            if chunk:
                chunks.append({'text': chunk, 'source': doc['source']})
            start += chunk_size - overlap
    return chunks

chunks_fixed = chunk_fixed(documents, chunk_size=400, overlap=50)
avg_len_fixed = sum(len(c['text']) for c in chunks_fixed) / len(chunks_fixed)

print(f'Strategy 1 — Fixed-size (size=400, overlap=50)')
print(f'  Total chunks: {len(chunks_fixed)}')
print(f'  Average chunk length: {avg_len_fixed:.0f} characters')
print(f'  Sample chunk:')
print(f"  {chunks_fixed[0]['text'][:200]}...")

In [ ]:
# ── Strategy 2: Paragraph-aware chunking ──

def chunk_paragraph(documents, max_chunk=600, overlap=80):
    chunks = []
    for doc in documents:
        paragraphs = re.split(r'\n\s*\n', doc['text'])
        current = ''
        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            if len(current) + len(para) + 1 <= max_chunk:
                current = (current + '\n' + para).strip()
            else:
                if current:
                    chunks.append({'text': current, 'source': doc['source']})
                current = (current[-overlap:] + '\n' + para).strip() if current else para
        if current:
            chunks.append({'text': current, 'source': doc['source']})
    return chunks

chunks_para = chunk_paragraph(documents, max_chunk=600, overlap=80)
avg_len_para = sum(len(c['text']) for c in chunks_para) / len(chunks_para)

print(f'Strategy 2 — Paragraph-aware (max=600, overlap=80)')
print(f'  Total chunks: {len(chunks_para)}')
print(f'  Average chunk length: {avg_len_para:.0f} characters')
print(f'  Sample chunk:')
print(f"  {chunks_para[0]['text'][:200]}...")

### Chunking Strategy Comparison

| Strategy | Chunk Size | Overlap | Total Chunks | Avg Length |
|----------|-----------|---------|-------------|------------|
| Fixed-size | 400 chars | 50 chars | *(run cell above)* | *(run cell above)* |
| Paragraph-aware | 600 chars max | 80 chars | *(run cell above)* | *(run cell above)* |

**Chosen strategy: Paragraph-aware**

**Why:** A resume and SOQ are naturally organized into semantic paragraphs (education, experience, skills, projects). Splitting on paragraph boundaries preserves complete units of meaning so each retrieved chunk is self-contained and coherent. Fixed-size splitting can cut mid-sentence, mixing parts of different sections and reducing retrieval precision.

In [ ]:
# Use paragraph-aware chunking going forward
final_chunks = chunks_para
print(f'Final chunk count: {len(final_chunks)}')

---
<a id='4-embedding'></a>
## 4. Embedding and Vector Store

Using `sentence-transformers/all-MiniLM-L6-v2` (free, runs locally) via ChromaDB's embedding function.

In [ ]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name='resume_rag',
    embedding_function=ef,
)

collection.add(
    documents=[c['text'] for c in final_chunks],
    metadatas=[{'source': c['source']} for c in final_chunks],
    ids=[f'chunk_{i}' for i in range(len(final_chunks))],
)

print(f'Vector store created with {collection.count()} chunks')

In [ ]:
# ── Verify: similarity search ──
test_queries = [
    'Python programming skills',
    'data analytics experience',
    'machine learning projects',
]

for query in test_queries:
    results = collection.query(query_texts=[query], n_results=2)
    print(f'Query: "{query}"')
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        print(f"  [{meta['source']}] {doc[:120]}...")
    print()

---
<a id='5-retrieval'></a>
## 5. Retrieval Chain

Connecting the vector store to `Qwen/Qwen2.5-7B-Instruct` via HuggingFace Inference API (free tier).

In [ ]:
hf_client = InferenceClient(token=HF_TOKEN)

def retrieve(query, k=3):
    results = collection.query(query_texts=[query], n_results=k)
    docs = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]
    return docs, sources

def ask_llm(prompt, max_tokens=300):
    response = hf_client.chat_completion(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return response.choices[0].message.content.strip()

def rag_query(question, system_prompt, k=3):
    docs, sources = retrieve(question, k=k)
    context = '\n\n'.join(docs)
    full_prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    answer = ask_llm(full_prompt)
    return answer, docs, sources

# Sanity check
test_answer, test_docs, test_sources = rag_query(
    'What technical skills does this person have?',
    system_prompt='Answer questions about this person based only on the provided context.'
)
print('Retrieval chain works!')
print(f'Sources used: {test_sources}')
print(f'Answer: {test_answer[:300]}')

---
<a id='6-prompting'></a>
## 6. Prompt Engineering

**Note (Tier 2):** Prompt design is your own work. Write 3 iterations below, using the same test question each time to make improvement measurable.

**Consistent test question:** *"What experience does this person have with data analytics?"*

In [ ]:
EVAL_QUESTION = 'What experience does this person have with data analytics?'

# ── Iteration 1: Baseline ──
PROMPT_V1 = """TODO: Write your v1 prompt here."""

answer_v1, _, _ = rag_query(EVAL_QUESTION, PROMPT_V1)
print('=== Prompt v1 Output ===')
print(answer_v1)

**v1 → v2: What changed and why?**
> TODO

In [ ]:
# ── Iteration 2: Add grounding ──
PROMPT_V2 = """TODO: Write your v2 prompt here."""

answer_v2, _, _ = rag_query(EVAL_QUESTION, PROMPT_V2)
print('=== Prompt v2 Output ===')
print(answer_v2)

**v2 → v3: What changed and why?**
> TODO

In [ ]:
# ── Iteration 3: Final ──
PROMPT_V3 = """TODO: Write your final prompt here."""

answer_v3, _, _ = rag_query(EVAL_QUESTION, PROMPT_V3)
print('=== Prompt v3 Output (Final) ===')
print(answer_v3)

**Overall improvement from v1 to v3:**
> TODO: Describe the measurable difference in output quality.

In [ ]:
FINAL_PROMPT = PROMPT_V3
print('Final prompt set for evaluation.')

---
<a id='7-evaluation'></a>
## 7. Evaluation

10 questions across 4 categories. Scoring per question:
- **Retrieval quality:** Yes / Partial / No
- **Faithfulness:** Faithful / Partial / Hallucinated
- **Answer quality:** 1–5

**Note (Tier 2):** Fill in scores and analysis yourself — AI is prohibited from evaluation.

In [ ]:
TEST_QUESTIONS = [
    {'q': 'What degree(s) does this person hold?',                          'category': 'Factual'},
    {'q': 'What programming languages does this person know?',              'category': 'Factual'},
    {'q': 'Describe the Paris Agreement climate classifier project.',       'category': 'Factual'},
    {'q': "How does this person's ML experience apply to business problems?", 'category': 'Inference'},
    {'q': 'What type of role would suit this person best and why?',         'category': 'Inference'},
    {'q': "What is this person's salary expectation?",                      'category': 'Out-of-scope'},
    {'q': 'Does this person have 10 years of professional experience?',     'category': 'Out-of-scope'},
    {'q': "What is this person's home address?",                            'category': 'Out-of-scope'},
    {'q': 'What did this person write in their statement of qualifications?','category': 'Specificity'},
    {'q': 'What model did the BSAN6070 final project use and why?',         'category': 'Specificity'},
]

results = []
for item in TEST_QUESTIONS:
    answer, docs, sources = rag_query(item['q'], FINAL_PROMPT, k=3)
    results.append({'category': item['category'], 'question': item['q'],
                    'sources': sources, 'answer': answer})
    print(f"[{item['category']}] {item['q']}")
    print(f"  Sources: {sources}")
    print(f"  Answer: {answer[:200]}")
    print()

In [ ]:
import pandas as pd

# Fill in YOUR scores after reading each answer above
scores = [
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
    {'retrieval_quality': 'TODO', 'faithfulness': 'TODO', 'quality_score': 0},
]

eval_df = pd.DataFrame([
    {
        'Category': r['category'],
        'Question': r['question'][:55] + '...',
        'Sources': ', '.join(r['sources']),
        'Retrieval': s['retrieval_quality'],
        'Faithfulness': s['faithfulness'],
        'Score (1-5)': s['quality_score'],
    }
    for r, s in zip(results, scores)
])

eval_df

### Evaluation Analysis

**Where does the chatbot succeed?**
> TODO

**Where does it fail? Why?**
> TODO

**What would you improve?**
> TODO

---

## After Completing This Notebook

1. Copy `PROMPT_V3` → `streamlit_app.py` `SYSTEM_PROMPT`
2. Copy `chunk_paragraph` → `streamlit_app.py` `chunk_documents()`
3. Add 3 sample questions to `streamlit_app.py` `samples` list
4. Fill in your scores above, then update `evaluation/test_results.md`
5. Write `memo.md` and complete `ai_log.md` (8+ entries)

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option A*